> # ⚠️ Do not run this notebook unattended
>
> The last section, **`# Hardware validation`**, connects to the real board, sends the SHTC3
> commands on I2C bus 0, and has a step where you breathe on the sensor. *Run All* and `jlx run` reach that section.
>
> - **Safe to run on its own:** everything above `# Hardware validation`. It runs on the host
>   against a fake part, writes the module and its tests, and needs no board.
> - **Hardware section:** run it one cell at a time, reading the **What to do** line above
>   each cell first.

**Where each cell runs.** Everything above `%board connect` (the first cells of
`# Hardware validation`) runs in ordinary Python (CPython) on this computer, with no board: it
writes the fake part (`shtc3_fake.py`), the driver (`shtc3.py`) and their tests, and checks the
driver against `FakeSHTC3`, in real and in simulated time. After `%board connect`, cells run in
MicroPython on the board, except `%%host` cells, which run on the computer. That section
uploads the freshly written package, checks the real SHTC3 with a matching test for every unit
test above, then disconnects.

# Hardware and configuration
What this notebook was validated with. The same table opens `# Hardware validation`.

| | Host run (simulation) | Hardware run |
|---|---|---|
| **Runs on** | CPython 3.13.15 in the micropylab kernel (1.0.0a1), no board | Unexpected Maker FeatherS3 (ESP32-S3), MicroPython 1.28.0 (v1.28.0, 2026-04-06), USB `/dev/ttyACM0` |
| **I2C bus** | `FakeBus` | `machine.I2C(0)`, **bus 0**, the STEMMA QT connector (scl=9 sda=8, on-board pull-ups), 100 kHz (`bus_open("feathers3", 0)`) |
| **Mux** | none | **none**: the SHTC3 is always at `0x70`, the mux's own address |
| **Sensor** | `FakeSHTC3` at `0x70`: idle (as after power-up), 25 C, 50 %RH, ID `0x0807`, measures in 11.0 ms (normal) / 0.5 ms (low power) | SparkFun Qwiic SHTC3 at `0x70` |
| **Other parts used** | none | none: the SHTC3 was the only part connected to the board. The TMP102 cross-check (bus 1, mux channel 2) was skipped in this run; an earlier run with it connected read 23.43 C against 22.88 C |
| **Datasheet** | Sensirion SHTC3, version 4, December 2022 | same |
| **Result** | 24 unit tests pass (7 of them on the fake part, 2 in simulated time) | 35 of 35 hardware checks passed (2026-11-14) |

The simulation and the bench differ on purpose here: a mux in front of the SHTC3 would have been
the better test, but the bench's mux has the same address (`0x70`), so the SHTC3 sits alone on bus
0. The fake also refuses wake-up for 2 ms after a NACK (the real part: under 1 ms) so a test can
see it at ms resolution.

# TODO:
Nothing open.

Measured on the bench (2026-11-14). The `## Diagnostics` cells print each of these again:
- In sleep the part answers a scan and the wake-up command, and NACKs every other command.
- Right after such a NACKed command, wake-up is NACKed too, for under 1 ms (one retry ~1 ms
  later got through in every trial). The driver retries wake-up up to 3 times, 1 ms apart.
- A wake-up command to a part that is already awake is acknowledged; a read with no result
  waiting is NACKed.
- With a quiet bus the part answered after 11.0 ms (normal) and 0.5 ms (low power); the driver
  waits the datasheet maximums, 12.1 ms and 0.8 ms. On the ESP32 each NACKed transfer costs
  about 0.8 ms, so polling cannot time a measurement finely.
- Clock stretching works on the ESP32 (a stretched read took 12.9 ms); the driver still waits
  instead, which works on every port.
- Temperature 23.43 C against the TMP102's 22.88 C beside it.

Decided (2026-11-14): the SHTC3 is on **I2C bus 0** (STEMMA QT, scl=9 sda=8), the only part on
that bus, not behind the mux. It is always at `0x70`, the mux's own address, so it cannot
share the mux's segment. Datasheet: Sensirion SHTC3, version 4, December 2022
(`docs/datasheets/Datasheet_SHTC3.pdf`).

Decided (2026-11-16): `FakeSHTC3` moved out of `micropython_i2c.fakes` into this notebook
(`shtc3_fake.py`, next to the driver). `temp_c` and `rh` accept signals
(`micropython_i2c.sim.signals`), and a clock-stretched read sleeps out the rest of the
measurement instead of spinning, so the fake runs under `SimClock`.

# Python file
location: `src/micropython_i2c/devices/sensors/environmental/shtc3.py` (driver) and
`src/micropython_i2c/devices/sensors/environmental/shtc3_fake.py` (fake part, host only)

tests: `tests/test_shtc3.py` (both)

## header

In [1]:
%%extract ../../../src/micropython_i2c/devices/sensors/environmental/shtc3.py
"""Sensirion SHTC3 humidity and temperature sensor (datasheet version 4, December 2022).

Usage on the board::

    from micropython_i2c import I2CConnection
    from micropython_i2c.devices.sensors.environmental.shtc3 import SHTC3

    sht = SHTC3(I2CConnection(bus, 0x70, lock=bus_lock))
    t_c, rh = sht.measure()         # -> (23.73, 62.97)

Two classes:

* ``SHTC3Cmds``: datasheet facts only (commands, timings, the ID pattern), no methods.
  Bring-up uses it to drive ``CommandI2CBase`` directly against the part.
* ``SHTC3``: the driver.

Command-driven: every operation is a 16-bit command, and a read returns 16-bit words, each
followed by a CRC-8 (poly 0x31, init 0xFF, the ``CommandI2CBase`` default). The part sleeps
between operations: the driver wakes it, works, and always puts it back to sleep. In sleep it
answers only the wake-up command. While it measures it NACKs everything.

The address is fixed at ``0x70``, the TCA9548A's default address too: never put the two on
one segment.

Generated by ``device_notebooks/sensors/environmental/shtc3_devNB``: edit the notebook, not
this file.
"""

"Sensirion SHTC3 humidity and temperature sensor (datasheet version 4, December 2022).\n\nUsage on the board::\n\n    from micropython_i2c import I2CConnection\n    from micropython_i2c.drivers.shtc3 import SHTC3\n\n    sht = SHTC3(I2CConnection(bus, 0x70, lock=bus_lock))\n    t_c, rh = sht.measure()         # -> (23.73, 62.97)\n\nTwo classes:\n\n* ``SHTC3Cmds``: datasheet facts only (commands, timings, the ID pattern), no methods.\n  Bring-up uses it to drive ``CommandI2CBase`` directly against the part.\n* ``SHTC3``: the driver.\n\nCommand-driven: every operation is a 16-bit command, and a read returns 16-bit words, each\nfollowed by a CRC-8 (poly 0x31, init 0xFF, the ``CommandI2CBase`` default). The part sleeps\nbetween operations: the driver wakes it, works, and always puts it back to sleep. In sleep it\nanswers only the wake-up command. While it measures it NACKs everything.\n\nThe address is fixed at ``0x70``, the TCA9548A's default address too: never put the two on\none segment.

# Imports
## External
None: the driver runs on MicroPython and uses only this package.

## Internal
`CommandI2CBase` for commands and CRC-checked words; `sleep_us` from `compat` for the waits.

In [2]:
%%extract ../../../src/micropython_i2c/devices/sensors/environmental/shtc3.py -a
from micropython_i2c.compat import sleep_us
from micropython_i2c.node import I2CDevice
from micropython_i2c.protocol import CommandI2CBase

# Setup

Notebook-only: run the test cells in the notebook. `raise_on_error=True` makes a failing test
fail the run.

In [3]:
import ipytest

ipytest.autoconfig(raise_on_error=True)

# Test file
The test file header: imports only. It is written, not run.

In [4]:
%%writefile ../../../tests/test_shtc3.py
import pytest

from micropython_i2c import I2CConnection
from micropython_i2c.devices.sensors.environmental.shtc3 import SHTC3, SHTC3Cmds
from micropython_i2c.devices.sensors.environmental.shtc3_fake import FakeSHTC3
from micropython_i2c.sim.bus import FakeBus, FakeRegisterPart
from micropython_i2c.sim.clock import SimClock
from micropython_i2c.sim.signals import ramp

Overwriting ../tests/test_shtc3.py


Notebook-only: the imports the validation and test cells use. The classes under test
(`FakeSHTC3`, `SHTC3Cmds`, `SHTC3`) are the ones defined in this notebook, not the modules'.

In [5]:
import pytest

from micropython_i2c import I2CConnection
from micropython_i2c.sim.bus import FakeBus, FakeRegisterPart
from micropython_i2c.sim.clock import SimClock
from micropython_i2c.sim.signals import ramp

# Intent
**Purpose:** read relative humidity and temperature from a Sensirion SHTC3.

**Why it exists:** the SHTC3 has no registers. Every operation is a command sequence (wake up,
command, wait, read, sleep), a forgotten sleep costs 45 µA idle instead of 0.3 µA, and a read
at the wrong moment is NACKed rather than answered late.

**Used by:** the routing table and manager (`"driver": "SHTC3"`) and the bench (SparkFun Qwiic
SHTC3 on I2C bus 0, no mux).

**Not its job:** the general-call reset (it resets every part on the bus that supports it),
clock stretching (the driver waits instead), the heater (the SHTC3 has none).

```python
sht = SHTC3(I2CConnection(bus, 0x70, lock=lock))   # checks the ID, leaves it asleep
sht.measure()                     # (23.73, 62.97)  C, %RH
sht.measure(low_power=True)       # 0.8 ms instead of 12.1 ms, noisier
```

```mermaid
sequenceDiagram
    participant D as driver
    participant S as SHTC3
    D->>S: wake-up 0x3517
    Note over S: up to 240 µs
    D->>S: measure 0x7866 (T first, no clock stretching)
    Note over S: up to 12.1 ms, NACKs everything
    D->>S: read 6 bytes
    S-->>D: T MSB, T LSB, CRC, RH MSB, RH LSB, CRC
    D->>S: sleep 0xB098
```

Conversion (section 5.11): $T = -45 + 175\,S_T / 2^{16}$ °C, $RH = 100\,S_{RH} / 2^{16}$ %RH.

## What the module defines

| Name | What it is |
|---|---|
| `SHTC3Cmds` | datasheet facts: commands (all eight measurement commands), timings, ID pattern, CRC examples |
| `SHTC3` | the driver: identify on construction, `measure`, `read_id`, `wake`, `soft_reset`, codecs |

# Fake part
`FakeSHTC3` stands in for the real part on a `FakeBus`, so the driver's tests and the
simulated-time scenario run on the host. It is written to its own module next to the driver,
`shtc3_fake.py`, and never runs on the board.

## header

In [ ]:
%%extract ../../../src/micropython_i2c/devices/sensors/environmental/shtc3_fake.py
"""A fake Sensirion SHTC3 (datasheet version 4) for host tests and simulations.

Usage::

    from micropython_i2c.devices.sensors.environmental.shtc3_fake import FakeSHTC3
    from micropython_i2c.sim.bus import FakeBus
    from micropython_i2c.sim.signals import ramp

    bus = FakeBus()
    bus.attach(0x70, FakeSHTC3(temp_c=ramp(20.0, 26.0, 0, 3600), rh=ramp(60.0, 45.0, 0, 3600)))

Temperature and humidity are plain numbers or signals (``micropython_i2c.sim.signals``), read
when a measurement command arrives. Time comes from ``micropython_i2c.compat``, so inside
``with SimClock():`` the part runs in simulated time; a clock-stretched read sleeps out the rest
of the measurement instead of spinning.

Host only.

Generated by ``device_notebooks/sensors/environmental/shtc3_devNB``: edit the notebook, not
this file.
"""

## Imports
Timing from `compat` (so `SimClock` drives the measurement time) and `signals` (its `value`) to read a signal. The module is
imported, not `value` itself, so the name cannot clash with a variable in the notebook.

In [ ]:
%%extract ../../../src/micropython_i2c/devices/sensors/environmental/shtc3_fake.py -a
from micropython_i2c.compat import sleep_us, ticks_add, ticks_diff, ticks_us
from micropython_i2c.sim import signals

## Public names

In [ ]:
%%extract ../../../src/micropython_i2c/devices/sensors/environmental/shtc3_fake.py -a
__all__ = ["FakeSHTC3"]

## `FakeSHTC3`
What the fake models, and where each behaviour comes from:

| Behaviour | Source |
|---|---|
| Commands: wake-up `0x3517`, sleep `0xB098`, soft reset `0x805D`, read ID `0xEFC8`; anything else is NACKed | datasheet Tables 9, 10, 12, 14 |
| Eight measurement commands: normal or low power, temperature or humidity first, clock stretching or not | datasheet Table 11 |
| Words: $S_T = (T + 45)\,2^{16}/175$, $S_{RH} = RH\,2^{16}/100$, each followed by CRC-8 (poly `0x31`, init `0xFF`) | datasheet section 5.11, Table 16 |
| ID register `0x0807` by default (bits 11 and 5..0 identify an SHTC3) | datasheet Table 15 |
| Starts awake (idle), as after power-up | datasheet, power-up |
| In sleep only wake-up is acknowledged; every other command is NACKed | datasheet sleep mode; measured on the bench, 2026-11-14, `shtc3_devNB` diagnostics |
| Right after a NACKed command, wake-up is NACKed too for `REFUSE_MS` | measured on the bench, 2026-11-14, `shtc3_devNB` diagnostics (under 1 ms; the fake uses 2 ms so a test sees it at ms resolution) |
| A measurement lasts `MEAS_MS`: 11.0 ms normal, 0.5 ms low power; until it ends every transfer is NACKed | measured on the bench, 2026-11-14, `shtc3_devNB` diagnostics (datasheet Table 5 maximums: 12.1 ms and 0.8 ms) |
| A read after a clock-stretching command during the measurement waits for the result (SCL held low); the fake sleeps out the remaining time on the `compat` clock | datasheet Table 11; measured on the bench (a stretched read took 12.9 ms) |
| The result can be read once; a read with nothing waiting is NACKed | measured on the bench, 2026-11-14, `shtc3_devNB` diagnostics |
| A wake-up to a part that is already awake is acknowledged | measured on the bench, 2026-11-14, `shtc3_devNB` diagnostics |
| Soft reset drops a waiting result and leaves the part idle | datasheet section 5.7 |
| `temp_c` and `rh` are read when the measurement command arrives; words saturate at `0x0000`/`0xFFFF` | simplification (the real part samples during the measurement) |

For tests: `corrupt` is the number of upcoming reads to flip one bit in (a CRC failure), and
`commands` lists every acknowledged command. Not modelled: the general-call reset, the
measurement's noise and the low-power mode's poorer repeatability (add `noisy` to a signal for
that).

In [ ]:
%%extract ../../../src/micropython_i2c/devices/sensors/environmental/shtc3_fake.py -a
class FakeSHTC3:
    """An SHTC3 behind a ``FakeBus``: commands, sleep, measurement timing and CRCs.

    Args:
        temp_c: Temperature in C: a number or a signal ``f(t_s)``, read at each measurement.
        rh: Relative humidity in %RH: a number or a signal, read at each measurement.
        id_word: What the ID register reads.

    Attributes:
        temp_c, rh: As given; replace them at any time.
        asleep: True after the sleep command, until a wake-up.
        pending: The words the next read returns, or ``None``.
        stretch: True if the last measurement command uses clock stretching.
        busy_until: ``ticks_us`` when the measurement ends, or ``None``.
        refuse_until: ``ticks_us`` until which wake-up is NACKed, or ``None``.
        corrupt: Number of upcoming reads to flip one bit in.
        commands: Every acknowledged command, as an int.
    """

    WAKEUP, SLEEP, SOFT_RESET, READ_ID = 0x3517, 0xB098, 0x805D, 0xEFC8
    # opcode -> (low_power, rh_first, clock_stretch), Table 11
    MEASURE = {
        0x7CA2: (False, False, True), 0x5C24: (False, True, True),
        0x7866: (False, False, False), 0x58E0: (False, True, False),
        0x6458: (True, False, True), 0x44DE: (True, True, True),
        0x609C: (True, False, False), 0x401A: (True, True, False),
    }  # fmt: skip
    MEAS_MS = {False: 11.0, True: 0.5}  # measured; the datasheet maximums are 12.1 and 0.8
    REFUSE_MS = 2  # measured under 1 ms; longer so a test sees it at ms resolution

    def __init__(self, temp_c=25.0, rh=50.0, id_word=0x0807):
        self.temp_c = temp_c
        self.rh = rh
        self.id_word = id_word
        self.asleep = False
        self.pending = None
        self.stretch = False
        self.busy_until = None
        self.refuse_until = None
        self.corrupt = 0
        self.commands = []

    @staticmethod
    def crc8(data):
        """CRC-8 of ``data``: polynomial 0x31, seeded 0xFF (Table 16)."""
        crc = 0xFF
        for byte in data:
            crc ^= byte
            for _ in range(8):
                crc = ((crc << 1) ^ 0x31) & 0xFF if crc & 0x80 else (crc << 1) & 0xFF
        return crc

    def _left_us(self):
        """Microseconds until the measurement ends; 0 when none is running."""
        if self.busy_until is not None:
            left = ticks_diff(self.busy_until, ticks_us())
            if left > 0:
                return left
        self.busy_until = None
        return 0

    def _nack(self):
        raise OSError(19, "ENODEV")

    def raw_write(self, data):
        cmd = int.from_bytes(bytes(data), "big")
        if self.asleep and cmd != self.WAKEUP:
            self.refuse_until = ticks_add(ticks_us(), self.REFUSE_MS * 1000)
            self._nack()
        if self.refuse_until is not None:
            if ticks_diff(self.refuse_until, ticks_us()) > 0:
                self._nack()
            self.refuse_until = None
        if self._left_us():
            self._nack()
        if cmd not in (self.WAKEUP, self.SLEEP, self.SOFT_RESET, self.READ_ID, *self.MEASURE):
            self._nack()  # not a command
        self.commands.append(cmd)
        if cmd == self.WAKEUP:
            self.asleep = False
        elif cmd == self.SLEEP:
            self.asleep = True
        elif cmd == self.SOFT_RESET:
            self.pending = None
        elif cmd == self.READ_ID:
            self.pending, self.stretch = [self.id_word], False
        elif cmd in self.MEASURE:
            low_power, rh_first, self.stretch = self.MEASURE[cmd]
            temp_c, rh = signals.value(self.temp_c), signals.value(self.rh)  # sampled now
            t = max(0, min(0xFFFF, int(round((temp_c + 45) * 65536 / 175))))
            h = max(0, min(0xFFFF, int(round(rh * 65536 / 100))))
            self.pending = [h, t] if rh_first else [t, h]
            self.busy_until = ticks_add(ticks_us(), int(self.MEAS_MS[low_power] * 1000))

    def raw_read(self, nbytes):
        if self.asleep:
            self._nack()
        left = self._left_us()
        if left:
            if not self.stretch:
                self._nack()
            sleep_us(left)  # clock stretching: SCL held low until the result is ready
            self.busy_until = None
        if self.pending is None:
            self._nack()  # nothing to read
        out = bytearray()
        for w in self.pending:
            pair = bytes([w >> 8, w & 0xFF])
            out += pair + bytes([self.crc8(pair)])
        self.pending = None
        if self.corrupt:
            out[0] ^= 0x01
            self.corrupt -= 1
        return bytes(out[:nbytes])

### validation/test
Two helpers shared by the validation cells and the test file: one sends a command as the bus
would, the other splits a read into words and checks every CRC.

In [ ]:
%%extract ../../../tests/test_shtc3.py -a
def send(part, cmd):
    """Write the 16-bit command ``cmd`` to ``part``, MSB first."""
    part.raw_write(bytes([cmd >> 8, cmd & 0xFF]))


def words_of(packet):
    """The 16-bit words of a read, each CRC checked (``AssertionError`` if one is wrong)."""
    words = []
    for i in range(0, len(packet), 3):
        assert FakeSHTC3.crc8(packet[i : i + 2]) == packet[i + 2], "bad CRC at byte {}".format(i)
        words.append((packet[i] << 8) | packet[i + 1])
    return words

#### validation: nominal usage
One measurement by hand in simulated time: the measurement command, a wait of `MEAS_MS`, the
read. Then a clock-stretched measurement whose read waits by itself, and the ID.

In [ ]:
with SimClock() as clock:
    part = FakeSHTC3(temp_c=23.7, rh=63.0)
    send(part, 0x7866)  # normal, temperature first, no clock stretching
    clock.advance(part.MEAS_MS[False] / 1000)
    t_word, rh_word = words_of(part.raw_read(6))
    print("T word {:#06x} -> {:.3f} C | RH word {:#06x} -> {:.3f} %RH".format(
        t_word, -45 + 175 * t_word / 65536, rh_word, 100 * rh_word / 65536))  # fmt: skip
    start = clock.now_us
    send(part, 0x5C24)  # normal, humidity first, clock stretching
    words_of(part.raw_read(6))
    print("stretched read waited {} us of simulated time".format(clock.now_us - start))
    send(part, part.READ_ID)
    print("ID: {:#06x} | commands: {}".format(words_of(part.raw_read(3))[0],
                                              [hex(c) for c in part.commands]))  # fmt: skip

#### validation: anti-nominal usage
A sleeping part NACKs every command but wake-up.

In [ ]:
part = FakeSHTC3()
send(part, part.SLEEP)
try:
    send(part, part.READ_ID)  # asleep: only wake-up is acknowledged
except OSError as err:
    print("raised OSError, as expected:\n    {}".format(err))
else:
    raise AssertionError("a command to a sleeping part should have raised OSError")

A read during a measurement without clock stretching is NACKed.

In [ ]:
with SimClock():
    part = FakeSHTC3()
    send(part, 0x7866)
    try:
        part.raw_read(6)  # still measuring, and the command does not stretch the clock
    except OSError as err:
        print("raised OSError, as expected:\n    {}".format(err))
    else:
        raise AssertionError("a read during the measurement should have raised OSError")

A word that is not a command is NACKed.

In [ ]:
part = FakeSHTC3()
try:
    send(part, 0x1234)  # not in the command tables
except OSError as err:
    print("raised OSError, as expected:\n    {}".format(err))
else:
    raise AssertionError("an unknown command should have raised OSError")

#### unit

With the words of Figure 7 as its temperature and humidity, a humidity-first measurement sends
the figure's packet byte for byte, CRCs included.

In [ ]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


def test_fake_reproduces_figure_7():
    with SimClock() as clock:
        part = FakeSHTC3(temp_c=-45 + 175 * 0x648B / 65536, rh=100 * 0xA133 / 65536)
        send(part, 0x58E0)  # normal, humidity first, no clock stretching
        clock.advance(0.011)
        assert part.raw_read(6) == b"\xa1\x33\x1c\x64\x8b\xc7"

Every measurement command returns both words in its order, and the words decode back to the
values (section 5.11) within one LSB.

In [ ]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


def test_fake_measurement_commands_and_words():
    with SimClock() as clock:
        part = FakeSHTC3(temp_c=-10.0, rh=5.0)
        for cmd, (_, rh_first, _) in FakeSHTC3.MEASURE.items():
            send(part, cmd)
            clock.advance(0.011)
            words = words_of(part.raw_read(6))
            t_word, rh_word = words[::-1] if rh_first else words
            assert abs(-45 + 175 * t_word / 65536 - -10.0) <= 175 / 65536
            assert abs(100 * rh_word / 65536 - 5.0) <= 100 / 65536

A clock-stretched read during the measurement waits for the result: it sleeps out exactly the
rest of `MEAS_MS` on the simulated clock (no busy-wait).

In [ ]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


def test_fake_clock_stretched_read_waits_for_the_result():
    with SimClock() as clock:
        part = FakeSHTC3()
        for cmd, low_power in ((0x7CA2, False), (0x6458, True)):
            start = clock.now_us
            send(part, cmd)
            clock.advance(0.0002)
            words_of(part.raw_read(6))
            assert clock.now_us - start == int(part.MEAS_MS[low_power] * 1000)

Without clock stretching, a read is NACKed until `MEAS_MS` has passed, then answered.

In [ ]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


def test_fake_nacks_until_the_measurement_ends():
    with SimClock() as clock:
        part = FakeSHTC3()
        send(part, 0x7866)
        clock.advance(0.0109)
        with pytest.raises(OSError):
            part.raw_read(6)
        with pytest.raises(OSError):  # a command too
            send(part, part.WAKEUP)
        clock.advance(0.0001)
        assert words_of(part.raw_read(6)) == [0x6666, 0x8000]  # 25 C, 50 %RH

In sleep only wake-up is acknowledged; right after a NACKed command wake-up is NACKed too, for
`REFUSE_MS`.

In [ ]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


def test_fake_sleep_and_the_refusal_after_a_nack():
    with SimClock() as clock:
        part = FakeSHTC3()
        send(part, part.SLEEP)
        with pytest.raises(OSError):
            send(part, part.READ_ID)
        with pytest.raises(OSError):
            send(part, part.WAKEUP)
        clock.advance(part.REFUSE_MS / 1000)
        send(part, part.WAKEUP)
        assert not part.asleep
        send(part, part.WAKEUP)  # already awake: acknowledged

The result is read once; a second read, or a read after a soft reset, is NACKed.

In [ ]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


def test_fake_result_is_read_once():
    part = FakeSHTC3()
    send(part, part.READ_ID)
    assert words_of(part.raw_read(3)) == [0x0807]
    with pytest.raises(OSError):
        part.raw_read(3)
    send(part, part.READ_ID)
    send(part, part.SOFT_RESET)
    with pytest.raises(OSError):
        part.raw_read(3)

Temperature and humidity signals are read when the measurement command arrives.

In [ ]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


def test_fake_reads_signals_at_the_measurement_command():
    with SimClock() as clock:
        part = FakeSHTC3(temp_c=ramp(20.0, 30.0, 0, 100), rh=ramp(60.0, 40.0, 0, 100))
        clock.advance(50)
        send(part, 0x7CA2)
        t_word, rh_word = words_of(part.raw_read(6))
        assert abs(-45 + 175 * t_word / 65536 - 25.0) <= 175 / 65536
        assert abs(100 * rh_word / 65536 - 50.0) <= 100 / 65536

# Code
## TODO:
Nothing

## Public names

In [6]:
%%extract ../../../src/micropython_i2c/devices/sensors/environmental/shtc3.py -a
__all__ = ["SHTC3", "SHTC3Cmds"]

## `SHTC3Cmds`
Every number here is from the datasheet. `CMD_MEASURE` maps `(low_power, rh_first,
clock_stretch)` to the opcode (Table 11), so the driver picks a command by meaning, not by hex.
The wake-up time is the power-up time `t_PU` (Figure 7 points there for it).

In [7]:
%%extract ../../../src/micropython_i2c/devices/sensors/environmental/shtc3.py -a
class SHTC3Cmds(CommandI2CBase, I2CDevice):
    """Datasheet facts for the SHTC3, with no driver logic."""

    ALLOWED_ADDRS = (0x70,)  # fixed (Table 8)
    ADDR_DEFAULT = 0x70
    BAUD_NOMINAL = 400_000
    BAUD_MAX = 1_000_000  # Fast-mode Plus (section 3.2)
    BAUD_MIN = 10_000

    # Commands (Tables 9, 10, 12, 14).
    CMD_WAKEUP = 0x3517
    CMD_SLEEP = 0xB098
    CMD_SOFT_RESET = 0x805D
    CMD_READ_ID = 0xEFC8

    # Measurement commands (Table 11): (low_power, rh_first, clock_stretch) -> opcode. Each
    # one measures both temperature and humidity; the flags set the read order.
    CMD_MEASURE = {
        (False, False, True): 0x7CA2,
        (False, True, True): 0x5C24,
        (False, False, False): 0x7866,
        (False, True, False): 0x58E0,
        (True, False, True): 0x6458,
        (True, True, True): 0x44DE,
        (True, False, False): 0x609C,
        (True, True, False): 0x401A,
    }

    # Timing (Table 5), microseconds, maximum values.
    T_WAKEUP_US = 240  # power-up time t_PU, also the wake-up time
    T_SOFT_RESET_US = 240
    T_MEAS_US = {False: 12_100, True: 800}  # normal, low power (one part: 11.0 and 0.5 ms)

    # Right after a NACKed command, wake-up is NACKed too for under 1 ms (measured): retry.
    WAKE_TRIES = 3
    WAKE_RETRY_US = 1000

    # ID register (Table 15): bits 11 and 5..0 identify the SHTC3; the rest vary per part.
    ID_MASK = 0x083F
    ID_SHTC3 = 0x0807

    # CRC (Table 16): CommandI2CBase's defaults are Sensirion's. The datasheet's examples:
    CRC_VECTORS = ((b"\x00", 0xAC), (b"\xbe\xef", 0x92))
    # Figure 7: a humidity-first read, 63 %RH and 23.7 C as sent on the wire.
    EXAMPLE_PACKET = b"\xa1\x33\x1c\x64\x8b\xc7"

    TEMP_MIN = -40.0  # specified range (Table 2)
    TEMP_MAX = 125.0

### validation/test
A helper that puts a `FakeSHTC3` (idle, as after power-up, 25 C and 50 %RH) at `0x70` on a fake
bus and binds a class to it. It returns the object and the fake part, and is shared by the
validation cells and the test file.

In [8]:
%%extract ../../../tests/test_shtc3.py -a
def on_fake(cls, **part_kw):
    """``(dev, part)``: ``cls`` bound to a fresh ``FakeSHTC3`` at 0x70 on a fake bus."""
    bus = FakeBus()
    part = bus.attach(0x70, FakeSHTC3(**part_kw))
    return cls(I2CConnection(bus, 0x70)), part

#### validation: nominal usage
A measurement by hand through `CommandI2CBase`: wake up, measure, wait the maximum time, read
two CRC-checked words, sleep.

In [9]:
cmds, part = on_fake(SHTC3Cmds)
cmds.cmd_send(cmds.CMD_WAKEUP)
sleep_us(cmds.T_WAKEUP_US)
cmds.cmd_send(cmds.CMD_MEASURE[(False, False, False)])
sleep_us(cmds.T_MEAS_US[False])
t_word, rh_word = cmds.words_read(2)
cmds.cmd_send(cmds.CMD_SLEEP)
print(f"T word {t_word:#06x}, RH word {rh_word:#06x} | commands sent:",
      [hex(c) for c in part.commands], "| asleep:", part.asleep)  # fmt: skip

T word 0x6666, RH word 0x8000 | commands sent: ['0x3517', '0x7866', '0xb098'] | asleep: True


#### validation: anti-nominal usage
A read straight after the measurement command, without the wait, is NACKed: the part does not
answer while it measures.

In [10]:
cmds, part = on_fake(SHTC3Cmds)
cmds.cmd_send(cmds.CMD_MEASURE[(False, False, False)])
try:
    cmds.words_read(2)  # no wait: still measuring
except OSError as err:
    print(f"raised OSError, as expected:\n    {err}")
else:
    raise AssertionError("a read during the measurement should have raised OSError")

raised OSError, as expected:
    [Errno 19] ENODEV


#### unit

The CRC parameters reproduce both examples in Table 16.

In [11]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


def test_crc_vectors_match_datasheet():
    for data, want in SHTC3Cmds.CRC_VECTORS:
        assert SHTC3Cmds.crc(data) == want

.                                                                                            [100%]
========================================= warnings summary =========================================
../.venv/lib/python3.13/site-packages/anyio/pytest_plugin.py:14
  /home/nonyourbusiness/Insync/string12oil@gmail.com/Google Drive/Project_restart/micropython-i2c-toolkit/.venv/lib/python3.13/site-packages/anyio/pytest_plugin.py:14: PytestRemovedIn10Warning: _pytest.python.CallSpec2 has been renamed to CallSpec.
  The CallSpec2 alias will be removed in pytest 10.
  Update imports to use CallSpec instead.
  See https://docs.pytest.org/en/stable/deprecations.html#callspec2-renamed
    from _pytest.python import CallSpec2

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
1 passed, 1 warning in 0.05s


The measurement command table is Table 11: eight distinct opcodes, and the driver's default
(normal mode, temperature first, no clock stretching) is `0x7866`.

In [12]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


def test_measurement_command_table():
    assert len(set(SHTC3Cmds.CMD_MEASURE.values())) == 8
    assert SHTC3Cmds.CMD_MEASURE[(False, False, False)] == 0x7866
    assert SHTC3Cmds.CMD_MEASURE[(True, True, True)] == 0x44DE

.                                                                                            [100%]
1 passed in 0.03s


Figure 7's packet decodes: both CRCs check, and the humidity-first words give 63 %RH and
23.7 C.

In [13]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


def test_datasheet_example_packet_decodes():
    p = SHTC3Cmds.EXAMPLE_PACKET
    assert SHTC3Cmds.crc(p[0:2]) == p[2] and SHTC3Cmds.crc(p[3:5]) == p[5]
    rh_word, t_word = (p[0] << 8) | p[1], (p[3] << 8) | p[4]
    # Section 5.11's formulas, written out (the driver's codecs are tested below).
    assert 100 * rh_word / 65536 == pytest.approx(63.0, abs=0.05)
    assert -45 + 175 * t_word / 65536 == pytest.approx(23.7, abs=0.05)

.                                                                                            [100%]
1 passed in 0.03s


Only `0x70` is allowed, and Fast-mode Plus is the clock ceiling.

In [14]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


def test_address_and_clock_limits():
    assert SHTC3Cmds.addr_ok(0x70) and not SHTC3Cmds.addr_ok(0x71)
    assert SHTC3Cmds.baud_ok(1_000_000) and not SHTC3Cmds.baud_ok(3_400_000)

.                                                                                            [100%]
1 passed in 0.02s


A word with a flipped bit fails its CRC, naming the word.

In [15]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


def test_words_read_rejects_a_bad_crc():
    cmds, part = on_fake(SHTC3Cmds)
    cmds.cmd_send(cmds.CMD_READ_ID)
    part.corrupt = 1
    with pytest.raises(ValueError, match="word 0"):
        cmds.words_read(1)

.                                                                                            [100%]
1 passed in 0.03s


## `SHTC3`
The driver wraps every operation in wake-up ... sleep, and puts the part to sleep even when the
read fails. Wake-up is retried (up to 3 times, 1 ms apart): right after a NACKed command the
part refuses it for a moment. The constructor reads the ID, which also leaves the part asleep (the datasheet
says to send it to sleep after power-up), and refuses a part whose ID is not an SHTC3's. A CRC
failure on that read counts as "not an SHTC3" too.

In [16]:
%%extract ../../../src/micropython_i2c/devices/sensors/environmental/shtc3.py -a
class SHTC3(SHTC3Cmds):
    """SHTC3 driver.

    Raises:
        RuntimeError: From the constructor, if the part at the address is not an SHTC3.
    """

    def __init__(self, conn, **kwargs):
        super().__init__(conn, **kwargs)
        try:
            ident = self.read_id()
        except ValueError as e:  # a CRC failure: something else answered
            raise RuntimeError("not an SHTC3 at {:#04x} ({})".format(self.addr, e)) from None
        if ident & self.ID_MASK != self.ID_SHTC3:
            raise RuntimeError("not an SHTC3 at {:#04x} (ID {:#06x})".format(self.addr, ident))
        self.ident = ident

    # Codecs (section 5.11).

    @staticmethod
    def celsius_from_raw(word):
        """Temperature word -> C."""
        return -45 + 175 * word / 65536

    @staticmethod
    def humidity_from_raw(word):
        """Humidity word -> %RH."""
        return 100 * word / 65536

    @staticmethod
    def celsius_to_raw(celsius):
        """C -> temperature word.

        Raises:
            ValueError: If the word would not fit 16 bits.
        """
        word = int((celsius + 45) * 65536 / 175 + 0.5)
        if not 0 <= word <= 0xFFFF:
            raise ValueError("{} C not representable".format(celsius))
        return word

    @staticmethod
    def humidity_to_raw(rh):
        """%RH -> humidity word.

        Raises:
            ValueError: If ``rh`` is outside 0..100.
        """
        if not 0.0 <= rh <= 100.0:
            raise ValueError("{} %RH outside 0-100".format(rh))
        return min(0xFFFF, int(rh * 65536 / 100 + 0.5))

    # Operations. Each one wakes the part and always puts it back to sleep.

    def wake(self):
        """Send wake-up and wait for the part to come up.

        Retried up to ``WAKE_TRIES`` times, ``WAKE_RETRY_US`` apart: right after a NACKed
        command the part refuses wake-up too, for under 1 ms (measured).

        Raises:
            OSError: If every try is NACKed (nothing at the address, or it is measuring).
        """
        for attempt in range(self.WAKE_TRIES):
            try:
                self.cmd_send(self.CMD_WAKEUP)
                break
            except OSError:
                if attempt == self.WAKE_TRIES - 1:
                    raise
                sleep_us(self.WAKE_RETRY_US)
        sleep_us(self.T_WAKEUP_US)

    def _awake(self, command, wait_us, n_words):
        """Wake, send ``command``, wait ``wait_us``, read ``n_words`` (CRC-checked), sleep."""
        self.wake()
        try:
            self.cmd_send(command)
            sleep_us(wait_us)
            return self.words_read(n_words)
        finally:
            self.cmd_send(self.CMD_SLEEP)

    def read_id(self):
        """The 16-bit ID register."""
        return self._awake(self.CMD_READ_ID, 0, 1)[0]

    def identify(self):
        """True if the ID register matches the SHTC3 pattern."""
        return self.read_id() & self.ID_MASK == self.ID_SHTC3

    def measure_raw(self, low_power=False):
        """``(temperature_word, humidity_word)`` from one measurement.

        Waits the maximum measurement time instead of using clock stretching.
        """
        cmd = self.CMD_MEASURE[(low_power, False, False)]  # temperature first, no stretching
        t_word, rh_word = self._awake(cmd, self.T_MEAS_US[low_power], 2)
        return t_word, rh_word

    def measure(self, low_power=False):
        """``(temperature_C, humidity_pct)`` from one measurement.

        Args:
            low_power: 0.8 ms instead of 12.1 ms, with poorer repeatability.

        Raises:
            ValueError: On a CRC mismatch in either word (the part is still put to sleep).
        """
        t_word, rh_word = self.measure_raw(low_power)
        return self.celsius_from_raw(t_word), self.humidity_from_raw(rh_word)

    def soft_reset(self):
        """Software reset (section 5.7), then back to sleep."""
        self.wake()
        self.cmd_send(self.CMD_SOFT_RESET)
        sleep_us(self.T_SOFT_RESET_US)
        self.cmd_send(self.CMD_SLEEP)

### validation/test: codecs
The codec cases `(word, C, %RH)`, shared by the unit tests below and copied to the board by the
hardware section: both ends of the word, mid-scale, and Figure 7's temperature word.

In [17]:
%%extract ../../../tests/test_shtc3.py -a
CODEC_CASES = [(0x0000, -45.0, 0.0), (0x8000, 42.5, 50.0), (0xFFFF, 129.99733, 99.99847),
               (0x648B, 23.73055, 39.27460)]  # fmt: skip

#### validation: nominal usage

In [18]:
for word, t, rh in CODEC_CASES:
    print(f"{word:#06x}: {SHTC3.celsius_from_raw(word):9.4f} C (want {t}) | "
          f"{SHTC3.humidity_from_raw(word):8.4f} %RH (want {rh})")  # fmt: skip
print("25 C ->", hex(SHTC3.celsius_to_raw(25.0)), "| 50 %RH ->", hex(SHTC3.humidity_to_raw(50.0)))

0x0000:  -45.0000 C (want -45.0) |   0.0000 %RH (want 0.0)
0x8000:   42.5000 C (want 42.5) |  50.0000 %RH (want 50.0)
0xffff:  129.9973 C (want 129.99733) |  99.9985 %RH (want 99.99847)
0x648b:   23.7305 C (want 23.73055) |  39.2746 %RH (want 39.2746)
25 C -> 0x6666 | 50 %RH -> 0x8000


#### validation: anti-nominal usage
120 %RH does not exist.

In [19]:
try:
    SHTC3.humidity_to_raw(120.0)  # above 100 %RH
except ValueError as err:
    print(f"raised ValueError, as expected:\n    {err}")
else:
    raise AssertionError("120 %RH should have raised ValueError")

raised ValueError, as expected:
    120.0 %RH outside 0-100


#### unit

The codecs decode the cases, and encode then decode returns the value to within one LSB.

In [20]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


@pytest.mark.parametrize("word, t, rh", CODEC_CASES)
def test_codecs(word, t, rh):
    assert SHTC3.celsius_from_raw(word) == pytest.approx(t, abs=1e-4)
    assert SHTC3.humidity_from_raw(word) == pytest.approx(rh, abs=1e-4)
    assert SHTC3.celsius_from_raw(SHTC3.celsius_to_raw(t)) == pytest.approx(t, abs=175 / 65536)
    assert SHTC3.humidity_from_raw(SHTC3.humidity_to_raw(rh)) == pytest.approx(rh, abs=100 / 65536)

....                                                                                         [100%]
4 passed in 0.04s


Values outside what a word can hold are refused.

In [21]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


@pytest.mark.parametrize(
    "fn, value",
    [("celsius_to_raw", -50.0), ("celsius_to_raw", 131.0), ("humidity_to_raw", -1.0),
     ("humidity_to_raw", 101.0)],
)  # fmt: skip
def test_codecs_refuse_out_of_range(fn, value):
    with pytest.raises(ValueError):
        getattr(SHTC3, fn)(value)

....                                                                                         [100%]
4 passed in 0.04s


### validation/test: the driver on a fake part

#### validation: nominal usage

In [22]:
sht, part = on_fake(SHTC3, temp_c=23.7, rh=63.0)
print("ID", hex(sht.ident), "| identify:", sht.identify(), "| asleep after init:", part.asleep)
print("measure:", sht.measure(), "| low power:", sht.measure(low_power=True))
print("commands of the last measurement:", [hex(c) for c in part.commands[-3:]])

ID 0x807 | identify: True | asleep after init: True
measure: (23.701171875, 63.00048828125) | low power: (23.701171875, 63.00048828125)
commands of the last measurement: ['0x3517', '0x609c', '0xb098']


#### validation: anti-nominal usage
Something at `0x70` that is not an SHTC3 (here a register part returning zeros) is refused.

In [23]:
bus = FakeBus()
bus.attach(0x70, FakeRegisterPart())
try:
    SHTC3(I2CConnection(bus, 0x70))
except RuntimeError as err:
    print(f"raised RuntimeError, as expected:\n    {err}")
else:
    raise AssertionError("a part that is not an SHTC3 should have raised RuntimeError")

raised RuntimeError, as expected:
    not an SHTC3 at 0x70 (word 0: checksum fail: computed 0x81, received 0x00)


#### unit

The constructor identifies the part and leaves it asleep.

In [24]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


def test_identify_and_init_sleeps():
    sht, part = on_fake(SHTC3, id_word=0x5887)  # unspecified bits set
    assert sht.ident == 0x5887 and sht.identify()
    assert part.asleep and part.commands[-1] == SHTC3.CMD_SLEEP

.                                                                                            [100%]
1 passed in 0.05s


A wrong ID, or a part whose answer fails the CRC, is refused.

In [25]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


def test_init_refuses_a_part_that_is_not_an_shtc3():
    with pytest.raises(RuntimeError, match="ID 0x1234"):
        on_fake(SHTC3, id_word=0x1234)
    bus = FakeBus()
    bus.attach(0x70, FakeRegisterPart())
    with pytest.raises(RuntimeError, match="not an SHTC3"):
        SHTC3(I2CConnection(bus, 0x70))

.                                                                                            [100%]
1 passed in 0.03s


A normal measurement sends wake-up, `0x7866`, sleep, and decodes the fake's values.

In [26]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


def test_measure_normal():
    sht, part = on_fake(SHTC3, temp_c=23.7, rh=63.0)
    t, rh = sht.measure()
    assert t == pytest.approx(23.7, abs=0.01) and rh == pytest.approx(63.0, abs=0.01)
    assert part.commands[-3:] == [SHTC3.CMD_WAKEUP, 0x7866, SHTC3.CMD_SLEEP]
    assert part.asleep

.                                                                                            [100%]
1 passed in 0.04s


A low-power measurement sends `0x609C` and decodes the same values.

In [27]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


def test_measure_low_power():
    sht, part = on_fake(SHTC3, temp_c=-10.0, rh=5.0)
    t, rh = sht.measure(low_power=True)
    assert t == pytest.approx(-10.0, abs=0.01) and rh == pytest.approx(5.0, abs=0.01)
    assert part.commands[-2] == 0x609C

.                                                                                            [100%]
1 passed in 0.02s


Ten measurements in a row each wait long enough: none is NACKed.

In [28]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


def test_measure_waits_long_enough():
    sht, _ = on_fake(SHTC3)
    for _ in range(10):
        assert sht.measure() == pytest.approx((25.0, 50.0), abs=0.01)

.                                                                                            [100%]
1 passed in 0.15s


A CRC failure raises, and the part is still put to sleep.

In [29]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


def test_always_back_to_sleep_even_on_crc_error():
    sht, part = on_fake(SHTC3)
    part.corrupt = 1
    with pytest.raises(ValueError, match="checksum"):
        sht.measure()
    assert part.asleep and part.commands[-1] == SHTC3.CMD_SLEEP

.                                                                                            [100%]
1 passed in 0.04s


A sleeping part refuses every command but wake-up, and right after that refuses wake-up for a
moment too. The driver retries wake-up, so its operations still work.

In [30]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


def test_commands_while_asleep_are_refused():
    sht, part = on_fake(SHTC3)
    assert part.asleep
    with pytest.raises(OSError):
        sht.cmd_send(SHTC3.CMD_READ_ID)
    with pytest.raises(OSError):  # straight after the refused command
        sht.cmd_send(SHTC3.CMD_WAKEUP)
    assert sht.identify()  # wake() retries

.                                                                                            [100%]
1 passed in 0.04s


A soft reset leaves the part asleep and working.

In [31]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


def test_soft_reset_leaves_it_asleep():
    sht, part = on_fake(SHTC3)
    sht.soft_reset()
    assert part.commands[-2:] == [SHTC3.CMD_SOFT_RESET, SHTC3.CMD_SLEEP]
    assert part.asleep and sht.identify()

.                                                                                            [100%]
1 passed in 0.07s


### validation/test: in simulated time
The driver against a room that warms from 20 C to 26 C over an hour while its humidity falls
from 60 %RH to 45 %RH (two `ramp` signals), inside `with SimClock() as clock:`. `clock.advance`
moves simulated time, so the hour takes milliseconds; the driver's own waits (wake-up, the
measurement time) move it too.

#### validation: nominal usage
A measurement every 10 simulated minutes, and how much simulated time one measurement takes.

In [ ]:
with SimClock() as clock:
    sht, part = on_fake(SHTC3, temp_c=ramp(20.0, 26.0, 0, 3600), rh=ramp(60.0, 45.0, 0, 3600))
    for _ in range(7):
        start = clock.now_us
        t, rh = sht.measure()
        print("t = {:4.0f} min: {:6.2f} C {:6.2f} %RH | took {} us".format(
            start / 60e6, t, rh, clock.now_us - start))  # fmt: skip
        clock.advance(600)

#### validation: anti-nominal usage
Simulated time adds no failure path: the refusals above (a part that is not an SHTC3, a CRC
failure) behave the same in it. The fake's own timing refusals are checked under `# Fake part`.

#### unit

Over an hour the driver follows both signals to within one LSB at every measurement, in
normal and low-power mode.

In [ ]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


def test_driver_follows_a_warming_drying_room():
    with SimClock() as clock:
        sht, _ = on_fake(SHTC3, temp_c=ramp(20.0, 26.0, 0, 3600), rh=ramp(60.0, 45.0, 0, 3600))
        for _ in range(7):
            minute = clock.now_s() / 60
            for low_power in (False, True):
                t, rh = sht.measure(low_power=low_power)
                assert abs(t - (20.0 + 6.0 * minute / 60)) <= 0.01
                assert abs(rh - (60.0 - 15.0 * minute / 60)) <= 0.01
            clock.advance(600)

One measurement takes at least the wake-up time plus the datasheet's maximum measurement time
of simulated time: the driver waits instead of polling.

In [ ]:
%%extract ../../../tests/test_shtc3.py -a --strip-ipytest
%%ipytest


def test_measure_waits_the_datasheet_time_in_simulated_time():
    with SimClock() as clock:
        sht, part = on_fake(SHTC3)
        for low_power in (False, True):
            start = clock.now_us
            sht.measure(low_power=low_power)
            assert clock.now_us - start >= SHTC3.T_WAKEUP_US + SHTC3.T_MEAS_US[low_power]
            assert part.asleep

# Hardware validation

> ## ⚠️ STOP — do not run this section unattended
>
> **Everything below talks to the real board.** It connects to the FeatherS3, uploads the
> package, and sends SHTC3 commands on I2C bus 0. One step asks you to breathe on the sensor.
>
> - Do **not** use *Run All*, *Run All Below* or `jlx run` on this section and walk away.
> - Run it **one cell at a time**. Read the **What to do** line above each cell first.
> - Each cell puts the SHTC3 back to sleep when it finishes, even if it fails.
> - The section ends with `%board disconnect`. If you stop partway, run that cell yourself.

**Hardware and configuration** (same as the table at the top):

| | Host run (simulation) | Hardware run |
|---|---|---|
| **Runs on** | CPython 3.13.15 in the micropylab kernel (1.0.0a1), no board | Unexpected Maker FeatherS3 (ESP32-S3), MicroPython 1.28.0 (v1.28.0, 2026-04-06), USB `/dev/ttyACM0` |
| **I2C bus** | `FakeBus` | `machine.I2C(0)`, **bus 0**, the STEMMA QT connector (scl=9 sda=8, on-board pull-ups), 100 kHz (`bus_open("feathers3", 0)`) |
| **Mux** | none | **none**: the SHTC3 is always at `0x70`, the mux's own address |
| **Sensor** | `FakeSHTC3` at `0x70`: idle (as after power-up), 25 C, 50 %RH, ID `0x0807`, measures in 11.0 ms (normal) / 0.5 ms (low power) | SparkFun Qwiic SHTC3 at `0x70` |
| **Other parts used** | none | none: the SHTC3 was the only part connected to the board. The TMP102 cross-check (bus 1, mux channel 2) was skipped in this run; an earlier run with it connected read 23.43 C against 22.88 C |
| **Datasheet** | Sensirion SHTC3, version 4, December 2022 | same |
| **Result** | 24 unit tests pass (7 of them on the fake part, 2 in simulated time) | 35 of 35 hardware checks passed (2026-11-14) |

The simulation and the bench differ on purpose here: a mux in front of the SHTC3 would have been
the better test, but the bench's mux has the same address (`0x70`), so the SHTC3 sits alone on bus
0. The fake also refuses wake-up for 2 ms after a NACK (the real part: under 1 ms) so a test can
see it at ms resolution.

Layout: connect and set up, find the part, then **matching tests** (one per unit test above,
same name, run on the real part and on MicroPython), hardware-only checks, **diagnostics**
(they print what the part does and do not count towards the tally), then the tally and
disconnect.

## Connect and set up

Notebook-only: connect to the board. Only one notebook can own the board: if another notebook
still has it, run `%board disconnect` there first.

**What to do:** make sure the board is plugged in and no other notebook owns it.

In [32]:
%board connect

Connected /dev/ttyACM0: FeatherS3 with ESP32-S3 - micropython 1.28.0 (v1.28.0 on 2026-04-06)


Notebook-only: remove the package an earlier upload left on the board, and make sure `/lib`
exists, so the copy below lands exactly at `/lib/micropython_i2c`.

**What to do:** nothing.

In [33]:
import os


def rmtree(path):
    """Delete a board folder and everything in it."""
    for entry in os.ilistdir(path):
        child = path + "/" + entry[0]
        if entry[1] == 0x4000:  # folder
            rmtree(child)
        else:
            os.remove(child)
    os.rmdir(path)


try:
    rmtree("/lib/micropython_i2c")
    print("removed the old /lib/micropython_i2c")
except OSError:
    print("no /lib/micropython_i2c on the board yet")
try:
    os.mkdir("/lib")
except OSError:
    pass  # already there

removed the old /lib/micropython_i2c


Notebook-only: copy `src/micropython_i2c`, including the file this notebook just wrote, to
`/lib/micropython_i2c`. The copy runs on the host; the path is relative to this notebook's folder.

**What to do:** nothing. Takes a few seconds.

In [34]:
%%host
from pathlib import Path

pkg = next(p for p in (Path("../../../src"), Path("src")) if p.is_dir()) / "micropython_i2c"
%files copy -r {pkg} :/lib/micropython_i2c

copied ../src/micropython_i2c/__init__.py -> :/lib/micropython_i2c/__init__.py
copied ../src/micropython_i2c/bits.py -> :/lib/micropython_i2c/bits.py
copied ../src/micropython_i2c/boards.py -> :/lib/micropython_i2c/boards.py
copied ../src/micropython_i2c/compat.py -> :/lib/micropython_i2c/compat.py
copied ../src/micropython_i2c/connection.py -> :/lib/micropython_i2c/connection.py
copied ../src/micropython_i2c/crc.py -> :/lib/micropython_i2c/crc.py
copied ../src/micropython_i2c/debug.py -> :/lib/micropython_i2c/debug.py
copied ../src/micropython_i2c/drivers/__init__.py -> :/lib/micropython_i2c/drivers/__init__.py
copied ../src/micropython_i2c/drivers/qwiic_relay.py -> :/lib/micropython_i2c/drivers/qwiic_relay.py
copied ../src/micropython_i2c/drivers/shtc3.py -> :/lib/micropython_i2c/drivers/shtc3.py
copied ../src/micropython_i2c/drivers/tca9548a.py -> :/lib/micropython_i2c/drivers/tca9548a.py
copied ../src/micropython_i2c/drivers/tmag5273.py -> :/lib/micropython_i2c/drivers/tmag5273.py


['../src/micropython_i2c/__init__.py -> :/lib/micropython_i2c/__init__.py',
 '../src/micropython_i2c/bits.py -> :/lib/micropython_i2c/bits.py',
 '../src/micropython_i2c/boards.py -> :/lib/micropython_i2c/boards.py',
 '../src/micropython_i2c/compat.py -> :/lib/micropython_i2c/compat.py',
 '../src/micropython_i2c/connection.py -> :/lib/micropython_i2c/connection.py',
 '../src/micropython_i2c/crc.py -> :/lib/micropython_i2c/crc.py',
 '../src/micropython_i2c/debug.py -> :/lib/micropython_i2c/debug.py',
 '../src/micropython_i2c/drivers/__init__.py -> :/lib/micropython_i2c/drivers/__init__.py',
 '../src/micropython_i2c/drivers/qwiic_relay.py -> :/lib/micropython_i2c/drivers/qwiic_relay.py',
 '../src/micropython_i2c/drivers/shtc3.py -> :/lib/micropython_i2c/drivers/shtc3.py',
 '../src/micropython_i2c/drivers/tca9548a.py -> :/lib/micropython_i2c/drivers/tca9548a.py',
 '../src/micropython_i2c/drivers/tmag5273.py -> :/lib/micropython_i2c/drivers/tmag5273.py',
 '../src/micropython_i2c/drivers/tmp

Notebook-only: drop any modules an earlier run left on the board, so the new files load, then
import what the checks use and zero the tally.

**What to do:** nothing.

In [35]:
import sys
import time

for name in [m for m in sys.modules if m.startswith("micropython_i2c")]:
    del sys.modules[name]

from micropython_i2c import I2CConnection, default_lock
from micropython_i2c.boards import bus_open
from micropython_i2c.compat import sleep_ms, sleep_us
from micropython_i2c.debug import check, check_raises, check_reset, check_summary, info
from micropython_i2c.debug import check_counts  # noqa: F401  (read from the host with %eval)
from micropython_i2c.devices.sensors.environmental.shtc3 import SHTC3, SHTC3Cmds
from micropython_i2c.devices.sensors.environmental.tmp102 import TMP102

check_reset()

Notebook-only: the bus and helpers. `rest(dev)` puts the part back to sleep from any state
(mid-measurement too). `asleep(dev)` tells whether it is asleep by sending a command that is
not wake-up: a sleeping part NACKs it. `CorruptingBus` flips one bit in the next read, for the
CRC checks. `ReplyingBus` does the real read but hands back a fixed reply, standing in for a
part that is not an SHTC3. If the bench is rewired, change the constants.

**What to do:** nothing.

In [36]:
SHT_BUS = 0  # STEMMA QT, scl=9 sda=8: the SHTC3 alone
SHT_ADDR = 0x70
TMP_BUS, MUX_ADDR, TMP_ADDR = 1, 0x70, 0x48  # the TMP102, for the temperature check
TMP_PATH = [[MUX_ADDR, 2]]

bus = bus_open("feathers3", SHT_BUS, freq=100_000)
bus_lock = default_lock()


class CorruptingBus:
    """The bus, flipping one bit in the next ``corrupt`` reads from the SHTC3."""

    def __init__(self, inner):
        self.inner = inner
        self.corrupt = 0

    def __getattr__(self, name):
        return getattr(self.inner, name)

    def readfrom(self, addr, nbytes, *args):
        data = self.inner.readfrom(addr, nbytes, *args)
        if addr == SHT_ADDR and self.corrupt:
            self.corrupt -= 1
            data = bytes([data[0] ^ 0x01]) + data[1:]
        return data


class ReplyingBus:
    """The bus, but reads from the SHTC3 return ``reply`` (after doing the real read)."""

    def __init__(self, inner, reply):
        self.inner = inner
        self.reply = reply

    def __getattr__(self, name):
        return getattr(self.inner, name)

    def readfrom(self, addr, nbytes, *args):
        data = self.inner.readfrom(addr, nbytes, *args)
        return self.reply[:nbytes] if addr == SHT_ADDR else data


def sht_conn(b=None):
    """A connection to the SHTC3 on bus 0 (optionally on a wrapped bus)."""
    return I2CConnection(b or bus, SHT_ADDR, None, bus_lock)


def rest(dev):
    """Back to sleep from any state: wake it (waiting out a measurement), then sleep."""
    for _ in range(3):
        try:
            dev.cmd_send(dev.CMD_WAKEUP)
            break
        except OSError:
            sleep_ms(15)  # measuring: it NACKs until done
    sleep_us(dev.T_WAKEUP_US)
    dev.cmd_send(dev.CMD_SLEEP)


def asleep(dev):
    """True if the part NACKs a command other than wake-up (the sign of sleep)."""
    try:
        dev.cmd_send(dev.CMD_READ_ID)
    except OSError:
        return True
    rest(dev)  # it was awake (and now has an ID waiting): put it back to sleep
    return False

## Find the part
Bus 0 should answer at `0x70` only. If `0x70` is missing, check the Qwiic cable is in the
STEMMA QT port.

**What to do:** nothing.

In [37]:
seen = bus.scan()
print("bus {} answers at:".format(SHT_BUS), [hex(a) for a in seen])
check("SHTC3 ACKs at 0x70 on bus 0", SHT_ADDR in seen)
others = [a for a in seen if a != SHT_ADDR]
if others:
    info("also on bus 0: {}".format([hex(a) for a in others]))
cmds = SHTC3Cmds(sht_conn())
rest(cmds)

bus 0 answers at: ['0x70']
  [PASS] SHTC3 ACKs at 0x70 on bus 0          


## Matching tests: `SHTC3Cmds`
One cell per unit test above, with the same name. Tests that need no bus traffic run too: they
check the arithmetic on MicroPython itself.

**Matches** `test_crc_vectors_match_datasheet`.

**What to do:** nothing. No bus traffic.

In [38]:
check("test_crc_vectors_match_datasheet",
      all(SHTC3Cmds.crc(d) == want for d, want in SHTC3Cmds.CRC_VECTORS))  # fmt: skip

  [PASS] test_crc_vectors_match_datasheet     


True

**Matches** `test_measurement_command_table`.

**What to do:** nothing. No bus traffic.

In [39]:
check("test_measurement_command_table",
      len(set(SHTC3Cmds.CMD_MEASURE.values())) == 8
      and SHTC3Cmds.CMD_MEASURE[(False, False, False)] == 0x7866
      and SHTC3Cmds.CMD_MEASURE[(True, True, True)] == 0x44DE)  # fmt: skip

  [PASS] test_measurement_command_table       


True

**Matches** `test_datasheet_example_packet_decodes`, on MicroPython.

**What to do:** nothing. No bus traffic.

In [40]:
p = SHTC3Cmds.EXAMPLE_PACKET
rh_ex = SHTC3.humidity_from_raw((p[0] << 8) | p[1])
t_ex = SHTC3.celsius_from_raw((p[3] << 8) | p[4])
check("test_datasheet_example_packet_decodes",
      SHTC3Cmds.crc(p[0:2]) == p[2] and SHTC3Cmds.crc(p[3:5]) == p[5]
      and abs(rh_ex - 63.0) < 0.05 and abs(t_ex - 23.7) < 0.05,
      "{:.2f} %RH, {:.2f} C".format(rh_ex, t_ex))  # fmt: skip

  [PASS] test_datasheet_example_packet_decodes 62.97 %RH, 23.73 C


True

**Matches** `test_address_and_clock_limits`.

**What to do:** nothing. No bus traffic.

In [41]:
check("test_address_and_clock_limits",
      SHTC3Cmds.addr_ok(0x70) and not SHTC3Cmds.addr_ok(0x71)
      and SHTC3Cmds.baud_ok(1_000_000) and not SHTC3Cmds.baud_ok(3_400_000))  # fmt: skip

  [PASS] test_address_and_clock_limits        


True

**Matches** `test_words_read_rejects_a_bad_crc`: the real part's ID read verifies, and the same
read with one bit flipped on the way in fails, naming word 0.

**What to do:** nothing.

In [42]:
cbus = CorruptingBus(bus)
cmds = SHTC3Cmds(sht_conn(cbus))
try:
    cmds.cmd_send(cmds.CMD_WAKEUP)
    sleep_us(cmds.T_WAKEUP_US)
    cmds.cmd_send(cmds.CMD_READ_ID)
    good = cmds.words_read(1)[0]
    check("the real ID read verifies", good & cmds.ID_MASK == cmds.ID_SHTC3, "{:#06x}".format(good))
    cmds.cmd_send(cmds.CMD_READ_ID)
    cbus.corrupt = 1
    check_raises("test_words_read_rejects_a_bad_crc", ValueError, cmds.words_read, 1)
finally:
    rest(SHTC3Cmds(sht_conn()))

  [PASS] the real ID read verifies            0x0887
  [PASS] test_words_read_rejects_a_bad_crc    
          word 0: checksum fail: computed 0xaf, received 0x5b


## Matching tests: codecs

Notebook-only: the codec cases, copied to the board (it cannot import the test file, which
needs pytest). The cell after it checks the copy against the notebook's.

**What to do:** nothing.

In [43]:
CODEC_CASES = [(0x0000, -45.0, 0.0), (0x8000, 42.5, 50.0), (0xFFFF, 129.99733, 99.99847),
               (0x648B, 23.73055, 39.27460)]  # fmt: skip

Notebook-only: on the host, read the board's copy back with `%eval` and check it matches.

**What to do:** nothing.

In [44]:
%%host
board = get_ipython().run_line_magic("eval", "CODEC_CASES")
if board != [list(c) for c in CODEC_CASES]:
    raise AssertionError("the board's copy of CODEC_CASES differs from the notebook's")
print("board copy of CODEC_CASES matches the notebook's")

board copy of CODEC_CASES matches the notebook's


**Matches** `test_codecs`.

**What to do:** nothing. No bus traffic.

In [45]:
for word, t, rh in CODEC_CASES:
    ok = (abs(SHTC3.celsius_from_raw(word) - t) < 1e-4
          and abs(SHTC3.humidity_from_raw(word) - rh) < 1e-4
          and abs(SHTC3.celsius_from_raw(SHTC3.celsius_to_raw(t)) - t) <= 175 / 65536
          and abs(SHTC3.humidity_from_raw(SHTC3.humidity_to_raw(rh)) - rh) <= 100 / 65536)  # fmt: skip
    check("test_codecs[{:#06x}]".format(word), ok)

  [PASS] test_codecs[0x0000]                  
  [PASS] test_codecs[0x8000]                  
  [PASS] test_codecs[0xffff]                  
  [PASS] test_codecs[0x648b]                  


**Matches** `test_codecs_refuse_out_of_range`.

**What to do:** nothing. No bus traffic.

In [46]:
for fn, value in (("celsius_to_raw", -50.0), ("celsius_to_raw", 131.0),
                  ("humidity_to_raw", -1.0), ("humidity_to_raw", 101.0)):  # fmt: skip
    check_raises("test_codecs_refuse_out_of_range[{}-{}]".format(fn, value), ValueError,
                 getattr(SHTC3, fn), value)  # fmt: skip

  [PASS] test_codecs_refuse_out_of_range[celsius_to_raw--50.0] 
          -50.0 C not representable
  [PASS] test_codecs_refuse_out_of_range[celsius_to_raw-131.0] 
          131.0 C not representable
  [PASS] test_codecs_refuse_out_of_range[humidity_to_raw--1.0] 
          -1.0 %RH outside 0-100
  [PASS] test_codecs_refuse_out_of_range[humidity_to_raw-101.0] 
          101.0 %RH outside 0-100


## Matching tests: `SHTC3`
The driver on the real part. The fake holds exact values; the real part reads the room, so
these check plausible ranges and agreement between readings.

**Matches** `test_identify_and_init_sleeps`: the ID matches the SHTC3 pattern, and the part is
asleep after construction.

**What to do:** nothing.

In [47]:
sht = SHTC3(sht_conn())
try:
    print("ID {:#06x}".format(sht.ident))
    check(
        "test_identify_and_init_sleeps", sht.identify() and asleep(sht), "{:#06x}".format(sht.ident)
    )
finally:
    rest(sht)

ID 0x0887
  [PASS] test_identify_and_init_sleeps        0x0887


**Matches** `test_init_refuses_a_part_that_is_not_an_shtc3`. Nothing else is on the bus, so
`ReplyingBus` stands in for a wrong part: the real transfers happen, but the reply is swapped
for a wrong ID with a good CRC, then for zeros (a bad CRC). Both must be refused, and the real
part must be fine afterwards.

**What to do:** nothing.

In [48]:
wrong_id = bytes([0x12, 0x34, SHTC3Cmds.crc(b"\x12\x34")])
for name, reply in (("wrong ID", wrong_id), ("bad CRC", b"\x00\x00\x00")):
    check_raises("test_init_refuses_a_part_that_is_not_an_shtc3[{}]".format(name), RuntimeError,
                 SHTC3, sht_conn(ReplyingBus(bus, reply)))  # fmt: skip
rest(SHTC3Cmds(sht_conn()))
check("the real part is still an SHTC3", SHTC3(sht_conn()).identify())

  [PASS] test_init_refuses_a_part_that_is_not_an_shtc3[wrong ID] 
          not an SHTC3 at 0x70 (ID 0x1234)
  [PASS] test_init_refuses_a_part_that_is_not_an_shtc3[bad CRC] 
          not an SHTC3 at 0x70 (word 0: checksum fail: computed 0x81, received 0x00)
  [PASS] the real part is still an SHTC3      


True

**Matches** `test_measure_normal`: a plausible room reading, and the part asleep afterwards.

**What to do:** keep your hands and breath away from the sensor.

In [49]:
sht = SHTC3(sht_conn())
try:
    t, rh = sht.measure()
    print("{:.2f} C, {:.2f} %RH".format(t, rh))
    check("test_measure_normal: temperature plausible", 10.0 <= t <= 40.0, "{:.2f} C".format(t))
    check("test_measure_normal: humidity plausible", 5.0 <= rh <= 95.0, "{:.2f} %RH".format(rh))
    check("test_measure_normal: asleep after", asleep(sht))
finally:
    rest(sht)

23.45 C, 38.10 %RH
  [PASS] test_measure_normal: temperature plausible 23.45 C
  [PASS] test_measure_normal: humidity plausible 38.10 %RH
  [PASS] test_measure_normal: asleep after    


**Matches** `test_measure_low_power`: a low-power reading agrees with a normal one (low power
has poorer repeatability, so the tolerance is loose).

**What to do:** keep your hands and breath away from the sensor.

In [51]:
sht = SHTC3(sht_conn())
try:
    t_n, rh_n = sht.measure()
    t_lp, rh_lp = sht.measure(low_power=True)
    print(
        "normal {:.2f} C {:.2f} %RH | low power {:.2f} C {:.2f} %RH".format(t_n, rh_n, t_lp, rh_lp)
    )
    check("test_measure_low_power", abs(t_lp - t_n) < 1.0 and abs(rh_lp - rh_n) < 5.0)
finally:
    rest(sht)

normal 23.57 C 38.13 %RH | low power 23.92 C 38.29 %RH
  [PASS] test_measure_low_power               


**Matches** `test_measure_waits_long_enough`: ten measurements in a row, none NACKed, and they
agree with each other.

**What to do:** keep your hands and breath away from the sensor.

In [52]:
sht = SHTC3(sht_conn())
try:
    readings = [sht.measure() for _ in range(10)]
    ts, rhs = [r[0] for r in readings], [r[1] for r in readings]
    print("T {:.2f}..{:.2f} C, RH {:.2f}..{:.2f} %RH".format(min(ts), max(ts), min(rhs), max(rhs)))
    check("test_measure_waits_long_enough",
          max(ts) - min(ts) < 0.5 and max(rhs) - min(rhs) < 2.0)  # fmt: skip
finally:
    rest(sht)

T 23.55..23.63 C, RH 38.11..38.14 %RH
  [PASS] test_measure_waits_long_enough       


**Matches** `test_always_back_to_sleep_even_on_crc_error`: one bit flipped on the way in fails
the CRC, and the part is still asleep afterwards.

**What to do:** nothing.

In [53]:
cbus = CorruptingBus(bus)
sht = SHTC3(sht_conn(cbus))
try:
    cbus.corrupt = 1
    check_raises("CRC failure raised", ValueError, sht.measure)
    check("test_always_back_to_sleep_even_on_crc_error", asleep(SHTC3Cmds(sht_conn())))
finally:
    rest(sht)

  [PASS] CRC failure raised                   
          word 0: checksum fail: computed 0x38, received 0xcc
  [PASS] test_always_back_to_sleep_even_on_crc_error 


**Matches** `test_commands_while_asleep_are_refused`: the sleeping part NACKs the ID command
when it is sent without wake-up. A bare wake-up straight after is usually NACKed too (for under
1 ms), and the driver's `identify()`, which retries wake-up, still works.

**What to do:** nothing. An `OSError` inside a `PASS` line is the expected NACK.

In [54]:
sht = SHTC3(sht_conn())
try:
    check_raises("command while asleep NACKed", OSError, sht.cmd_send, sht.CMD_READ_ID)
    try:
        sht.cmd_send(sht.CMD_WAKEUP)
        info("a bare wake-up straight after was accepted this time")
    except OSError:
        print("a bare wake-up straight after: NACKed, as measured")
    check("test_commands_while_asleep_are_refused", sht.identify())
finally:
    rest(sht)

  [PASS] command while asleep NACKed          
          [Errno 19] ENODEV
a bare wake-up straight after: NACKed, as measured
  [PASS] test_commands_while_asleep_are_refused 


**Matches** `test_soft_reset_leaves_it_asleep`.

**What to do:** nothing.

In [55]:
sht = SHTC3(sht_conn())
try:
    sht.soft_reset()
    check("test_soft_reset_leaves_it_asleep", asleep(sht) and sht.identify())
finally:
    rest(sht)

  [PASS] test_soft_reset_leaves_it_asleep     


## Hardware-only checks

The measurement times, with a quiet bus: each trial waits a fixed time with no bus traffic, then
reads once (polling would cost about 0.8 ms per NACK on the ESP32). The cell finds the shortest
wait that works, and checks that the driver's waits (the datasheet maximums, 12.1 ms and
0.8 ms) work and are not shorter than that. It also checks that a read during the measurement
is NACKed.

**What to do:** nothing. Takes a few seconds.

In [56]:
cmds = SHTC3Cmds(sht_conn())
rest(cmds)


def works_after(low_power, wait_us):
    """One measurement with a fixed wait and a single read: True if the read is answered."""
    cmds.cmd_send(cmds.CMD_WAKEUP)
    sleep_us(cmds.T_WAKEUP_US)
    cmds.cmd_send(cmds.CMD_MEASURE[(low_power, False, False)])
    sleep_us(wait_us)
    try:
        cmds.words_read(2)
        ok = True
    except OSError:
        ok = False
        sleep_ms(15)  # let it finish
    rest(cmds)
    return ok


try:
    for low_power, waits in ((False, range(9000, 12600, 500)), (True, range(200, 900, 100))):
        name = "low power" if low_power else "normal"
        driver_us = cmds.T_MEAS_US[low_power]
        shortest = None
        for w in waits:
            if works_after(low_power, w):
                shortest = w
                break
        print("{}: shortest working wait {} | the driver waits {:.1f} ms".format(
            name, "none" if shortest is None else "{:.1f} ms".format(shortest / 1000),
            driver_us / 1000))  # fmt: skip
        check("{}: the driver's wait works".format(name), works_after(low_power, driver_us))
        check("{}: done within the datasheet maximum".format(name),
              shortest is not None and shortest <= driver_us)  # fmt: skip
    check("a read 0.1 ms into the measurement is NACKed", not works_after(False, 100))
finally:
    rest(cmds)

normal: shortest working wait 11.0 ms | the driver waits 12.1 ms
  [PASS] normal: the driver's wait works      
  [PASS] normal: done within the datasheet maximum 
low power: shortest working wait 0.5 ms | the driver waits 0.8 ms
  [PASS] low power: the driver's wait works   
  [PASS] low power: done within the datasheet maximum 
  [PASS] a read 0.1 ms into the measurement is NACKed 


The SHTC3's temperature against the TMP102 on bus 1 (behind the mux, channel 2), when that is
connected. Both are accurate parts (±0.2 C and ±0.5 C typical). Skipped (reported, not failed)
when the mux does not answer on bus 1.

**What to do:** if the TMP102 is connected, keep your fingers off both sensors for a minute
first.

In [57]:
bus1 = bus_open("feathers3", TMP_BUS, freq=100_000)
if MUX_ADDR not in bus1.scan():
    info("no mux on bus {}: the TMP102 is not connected, so no cross-check".format(TMP_BUS))
else:
    sht = SHTC3(sht_conn())
    try:
        t_sht, _ = sht.measure()
        t_tmp = TMP102(I2CConnection(bus1, TMP_ADDR, TMP_PATH)).temperature()
        print("SHTC3 {:.2f} C | TMP102 {:.2f} C".format(t_sht, t_tmp))
        check("SHTC3 agrees with the TMP102", abs(t_sht - t_tmp) < 1.5,
              "{:+.2f} C apart".format(t_sht - t_tmp))  # fmt: skip
    finally:
        rest(sht)

  [info] no mux on bus 1: the TMP102 is not connected, so no cross-check


The humidity responds to a real change. The cell samples for 10 seconds and prints a reading
every second.

**What to do:** run the cell, and when it prints **breathe now**, breathe gently on the sensor
from a few centimetres, two or three times. No time? Skip this cell.

In [58]:
sht = SHTC3(sht_conn())
try:
    rhs = [sht.measure()[1]]
    print("start {:.1f} %RH. Breathe now, for the next few seconds".format(rhs[0]))
    for i in range(10):
        sleep_ms(1000)
        rhs.append(sht.measure()[1])
        print("  {:>2} s: {:.1f} %RH".format(i + 1, rhs[-1]))
    print("RH from {:.1f} to {:.1f} %RH".format(min(rhs), max(rhs)))
    check("humidity rose with your breath", max(rhs) - rhs[0] > 5.0,
          "+{:.1f} %RH".format(max(rhs) - rhs[0]))  # fmt: skip
finally:
    rest(sht)

start 38.1 %RH. Breathe now, for the next few seconds
   1 s: 38.1 %RH
   2 s: 38.1 %RH
   3 s: 38.1 %RH
   4 s: 41.6 %RH
   5 s: 47.9 %RH
   6 s: 54.3 %RH
   7 s: 59.9 %RH
   8 s: 65.2 %RH
   9 s: 68.8 %RH
  10 s: 71.6 %RH
RH from 38.1 to 71.6 %RH
  [PASS] humidity rose with your breath       +33.5 %RH


## Diagnostics
These print what the part does in cases the datasheet leaves open; the TODO cell at the top
records what they showed on the bench. They do not count towards the tally. Rerun them on a
new part or board.

### Sleep: what does the part acknowledge?
A scan (address only), a command other than wake-up, and the wake-up command, each sent to the
sleeping part.

**What to do:** nothing.

In [59]:
cmds = SHTC3Cmds(sht_conn())
rest(cmds)
try:
    print("scan while asleep sees 0x70:", SHT_ADDR in bus.scan())
    for name, cmd in (("read-ID command", cmds.CMD_READ_ID), ("wake-up command", cmds.CMD_WAKEUP)):
        try:
            cmds.cmd_send(cmd)
            print("{} while asleep: acknowledged".format(name))
        except OSError as e:
            print("{} while asleep: NACK ({})".format(name, e))
        rest(cmds)
finally:
    rest(cmds)

scan while asleep sees 0x70: True
read-ID command while asleep: NACK ([Errno 19] ENODEV)
wake-up command while asleep: acknowledged


### Wake-up when awake, and a read with nothing pending
The fake assumes the wake-up command is acknowledged by a part that is already awake, and that
a read with no result pending is NACKed.

**What to do:** nothing.

In [60]:
cmds = SHTC3Cmds(sht_conn())
rest(cmds)
try:
    cmds.cmd_send(cmds.CMD_WAKEUP)
    sleep_us(cmds.T_WAKEUP_US)
    try:
        cmds.cmd_send(cmds.CMD_WAKEUP)
        print("second wake-up (part awake): acknowledged")
    except OSError as e:
        print("second wake-up (part awake): NACK ({})".format(e))
    try:
        raw = cmds.read_raw(3)
        print("read with nothing pending: answered", " ".join("{:02x}".format(b) for b in raw))
    except OSError as e:
        print("read with nothing pending: NACK ({})".format(e))
finally:
    rest(cmds)

second wake-up (part awake): acknowledged
read with nothing pending: NACK ([Errno 19] ENODEV)


### Does a clock-stretching measurement work on this board?
The driver waits instead of using clock stretching. This tries the stretching command: the
read is sent at once, and the part holds SCL low for about 11 ms. The ESP32's I2C timeout may
cut that short.

**What to do:** nothing.

In [61]:
cmds = SHTC3Cmds(sht_conn())
try:
    cmds.cmd_send(cmds.CMD_WAKEUP)
    sleep_us(cmds.T_WAKEUP_US)
    cmds.cmd_send(cmds.CMD_MEASURE[(False, False, True)])
    start = time.ticks_us()
    try:
        t_word, rh_word = cmds.words_read(2)
        print("stretched read took {:.2f} ms: {:.2f} C, {:.2f} %RH".format(
            time.ticks_diff(time.ticks_us(), start) / 1000, SHTC3.celsius_from_raw(t_word),
            SHTC3.humidity_from_raw(rh_word)))  # fmt: skip
    except OSError as e:
        print("stretched read failed after {:.2f} ms: {}".format(
            time.ticks_diff(time.ticks_us(), start) / 1000, e))  # fmt: skip
finally:
    sleep_ms(15)
    rest(cmds)

stretched read took 13.02 ms: 25.18 C, 81.25 %RH


### How long after a refused command is wake-up refused?
Measured 2026-11-14: under 1 ms (one NACK, then accepted ~1 ms later, in 4 of 5 trials). Five
trials: put it to sleep, send the ID command (NACKed), then try wake-up every ~0.1 ms and time
how long until it is accepted.

**What to do:** nothing.

In [62]:
cmds = SHTC3Cmds(sht_conn())
rest(cmds)
try:
    for trial in range(5):
        rest(cmds)
        try:
            cmds.cmd_send(cmds.CMD_READ_ID)  # to the sleeping part: NACKed
            print("trial {}: the command was acknowledged (part awake?)".format(trial))
            continue
        except OSError:
            pass
        start, nacks = time.ticks_us(), 0
        while True:
            try:
                cmds.cmd_send(cmds.CMD_WAKEUP)
                break
            except OSError:
                nacks += 1
                if nacks > 500:
                    break
                sleep_us(100)
        took = time.ticks_diff(time.ticks_us(), start) / 1000
        print(
            "trial {}: wake-up accepted after {:.2f} ms ({} refused first)".format(
                trial, took, nacks
            )
        )
finally:
    rest(cmds)

trial 0: wake-up accepted after 1.09 ms (1 refused first)
trial 1: wake-up accepted after 1.01 ms (1 refused first)
trial 2: wake-up accepted after 0.99 ms (1 refused first)
trial 3: wake-up accepted after 1.02 ms (1 refused first)
trial 4: wake-up accepted after 1.01 ms (1 refused first)


### The shortest wait that works, every step
The full table behind the timing check: each trial waits a fixed time with no bus traffic, then
reads once. `ok` means the read was answered. Measured 2026-11-14: from 11.0 ms (normal) and
0.5 ms (low power). The driver waits 12.1 ms and 0.8 ms.

**What to do:** nothing. Takes a few seconds.

In [63]:
cmds = SHTC3Cmds(sht_conn())
rest(cmds)


def works_after(low_power, wait_us):
    """One measurement with a fixed wait and a single read: True if the read is answered."""
    cmds.cmd_send(cmds.CMD_WAKEUP)
    sleep_us(cmds.T_WAKEUP_US)
    cmds.cmd_send(cmds.CMD_MEASURE[(low_power, False, False)])
    sleep_us(wait_us)
    try:
        cmds.words_read(2)
        ok = True
    except OSError:
        ok = False
        sleep_ms(15)  # let it finish
    rest(cmds)
    return ok


try:
    for low_power, waits in ((False, range(8000, 12600, 500)), (True, range(200, 1300, 100))):
        row = [
            "{:.1f}:{}".format(w / 1000, "ok" if works_after(low_power, w) else "--") for w in waits
        ]
        print("{:<9} wait ms -> {}".format("low power" if low_power else "normal", "  ".join(row)))
finally:
    rest(cmds)

normal    wait ms -> 8.0:--  8.5:--  9.0:--  9.5:--  10.0:--  10.5:--  11.0:ok  11.5:ok  12.0:ok  12.5:ok
low power wait ms -> 0.2:--  0.3:--  0.4:--  0.5:ok  0.6:ok  0.7:ok  0.8:ok  0.9:ok  1.0:ok  1.1:ok  1.2:ok


### What does one read during a measurement do?
An early polling check once saw a low-power read answered only after 24 ms with no NACK. Not
reproduced: on 2026-11-14 one read 0.1 ms after each command was NACKed, and the NACK itself
took about 0.8 ms. This sends that read and times it.

**What to do:** nothing.

In [64]:
cmds = SHTC3Cmds(sht_conn())
rest(cmds)
try:
    for low_power in (False, True):
        cmds.cmd_send(cmds.CMD_WAKEUP)
        sleep_us(cmds.T_WAKEUP_US)
        cmds.cmd_send(cmds.CMD_MEASURE[(low_power, False, False)])
        sleep_us(100)  # well inside the measurement
        start = time.ticks_us()
        try:
            cmds.words_read(2)
            result = "answered"
        except OSError as e:
            result = "NACK ({})".format(e)
        took = time.ticks_diff(time.ticks_us(), start) / 1000
        print("{:<9} one read 0.1 ms after the command: {}, the read took {:.2f} ms".format(
            "low power" if low_power else "normal", result, took))  # fmt: skip
        sleep_ms(15)
        rest(cmds)
finally:
    rest(cmds)

normal    one read 0.1 ms after the command: NACK ([Errno 19] ENODEV), the read took 0.92 ms
low power one read 0.1 ms after the command: NACK ([Errno 19] ENODEV), the read took 0.75 ms


## Tally and disconnect

Notebook-only: the PASS/FAIL tally of every check in this section.

**What to do:** read it. Any `FAIL` line above says which check and why.

In [65]:
check_summary()

35 passed, 0 failed


(35, 0)

Notebook-only: copy the tally to the host before disconnecting, so the last cell can fail
the run after the board is released.

**What to do:** nothing.

In [66]:
%%host
hw_passed, hw_failed = get_ipython().run_line_magic("eval", "check_counts()")

Notebook-only: release the board, so the next run (or a kernel restart) starts on the host.

**What to do:** nothing.

In [67]:
%board disconnect

Disconnected /dev/ttyACM0 (port closed, board not reset)


Notebook-only: fail the run if any hardware check failed. It runs on the host, so it works
whether or not the board is still attached.

**What to do:** nothing. An error here means a `FAIL` line above.

In [68]:
%%host
if hw_failed:
    raise AssertionError("{} of {} hardware checks failed".format(hw_failed, hw_passed + hw_failed))
print("all {} hardware checks passed".format(hw_passed))

all 35 hardware checks passed
